# Textbooks Are All You Need: Microsoft PHI-1.5 Model with 1.3 Billion Parameters

Maintained runnable companion. Original source code is preserved verbatim in the nonexecuting appendix. Git notebooks are output-free. No API key or paid inference is needed.

In [ ]:
import sys, hashlib
from importlib.metadata import version
from pathlib import Path
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import hf_hub_download
torch.set_num_threads(4)
MODEL_ID = "microsoft/phi-1_5"
REVISION = "77aa61eeac94fbf33d492b9f2744c98b42d5b5eb"
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__, "Transformers:", version("transformers"))
print("Model:", MODEL_ID, "revision:", REVISION, "device:", device)

In [ ]:
weight_path = Path(hf_hub_download(MODEL_ID, "model.safetensors", revision=REVISION))
with weight_path.open("rb") as handle:
    weight_sha256 = hashlib.file_digest(handle, "sha256").hexdigest()
assert weight_sha256 == "b4f562cc4b12f1154a0b658b9d74ab9d024a686420fdf154ebc00a2219976df7"
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, revision=REVISION, trust_remote_code=False,
    dtype=torch.float32, attn_implementation="eager").to(device).eval()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=REVISION, trust_remote_code=False)
print("Verified checkpoint SHA-256:", weight_sha256)
print("Native model class:", type(model).__name__)
print("Parameters:", model.num_parameters())

In [ ]:
prompt = '```python\ndef print_prime(n):\n   """\n   Print all primes between 1 and n\n   """'
inputs = tokenizer(prompt, return_tensors="pt", return_attention_mask=False).to(device)
print("Prompt tokens:", inputs.input_ids.shape[1])
print(prompt)

In [ ]:
with torch.inference_mode():
    outputs = model.generate(**inputs, max_length=200, do_sample=False,
                             pad_token_id=tokenizer.eos_token_id)
text = tokenizer.batch_decode(outputs)[0]
print(text)

In [ ]:
assert type(model).__name__ == "PhiForCausalLM" and not model.training
assert 1_000_000_000 < model.num_parameters() < 1_500_000_000
prefix_length = inputs.input_ids.shape[1]
assert prefix_length < outputs.shape[1] <= 200
assert torch.equal(outputs[:, :prefix_length], inputs.input_ids)
assert text.startswith(prompt) and text == tokenizer.decode(outputs[0])
with torch.inference_mode():
    logits = model(**inputs).logits[:, -1, :]
assert torch.isfinite(logits).all()
expected_first_token = logits.argmax(dim=-1)
assert torch.equal(outputs[:, prefix_length], expected_first_token)
assert all(p.device.type == device for p in model.parameters())
print("PASS: official pinned weights/hash, native implementation, exact source prompt, bounded greedy inference, unchanged prefix and independent first-token argmax.")
print("Generated tokens:", outputs.shape[1] - prefix_length)
print("Generated code is displayed, not automatically executed.")

## Original source code — nonexecuting appendix

### Original block 1

```text
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

torch.set_default_device("cuda")

model = AutoModelForCausalLM.from_pretrained("microsoft/phi-1_5", trust_remote_code=True)

tokenizer = AutoTokenizer.from_pretrained("microsoft/phi-1_5", trust_remote_code=True)

inputs = tokenizer('''```python
def print_prime(n):
   """
   Print all primes between 1 and n
   """''', return_tensors="pt", return_attention_mask=False)

outputs = model.generate(**inputs, max_length=200)
text = tokenizer.batch_decode(outputs)[0]
print(text)
```